# 개업_점포_수 보정을 왜 안 했는지 재검증

02_EDA_상권쇠퇴분석.ipynb STEP 1에서 "개업_점포_수도 프랜차이즈처럼 %가 개수로 잘못 들어간 오류가 통계적으로 보이지만, 공식 보정을 적용하면 결과가 너무 크게 흔들려서(위험 상권 299개→439개) 클리핑만 적용하기로 결정"했던 근거를 다시 눈으로 확인한다.

핵심 질문: "문제 있는 행은 5.29%뿐인데 왜 결과 전체가 흔들리는가?" → 그 5.29%가 개업_점포_수 합계의 대부분을 차지하기 때문임을 실제 데이터로 보여준다.

In [1]:
import pandas as pd
import numpy as np

# stores 데이터는 cp949(euc-kr) 인코딩 (데이터_명세.md 참고)
df = pd.read_csv('../data/seoul_commercial_district_stores_2025.csv', encoding='cp949')
print(df.shape)
df.head()

(304775, 12)


,기준_년분기_코드,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,서비스_업종_코드,서비스_업종_코드_명,전체_점포_수,일반_점포_수,프랜차이즈_점포_수,개업_점포_수,폐업_점포_수
0,20251,A,골목상권,3110001,이북5도청사,CS100001,한식음식점,12,13,0,8,1
1,20251,A,골목상권,3110001,이북5도청사,CS100003,일식음식점,1,1,0,0,0
2,20251,A,골목상권,3110001,이북5도청사,CS100008,분식전문점,2,2,0,0,0
3,20251,A,골목상권,3110001,이북5도청사,CS100009,호프-간이주점,2,3,0,33,1
4,20251,A,골목상권,3110001,이북5도청사,CS100010,커피-음료,2,2,0,0,0


## STEP 1. 전체_점포_수_보정 재현 (02번 노트북과 동일 로직)

개업_점포_수가 "전체_점포_수를 넘는지" 판단하려면, 먼저 프랜차이즈 보정을 거친 신뢰할 수 있는 전체_점포_수_보정 값이 있어야 한다. 02번 노트북 STEP1 코드를 그대로 재사용.

In [2]:
# 프랜차이즈_점포_수 보정: 공식(전체=일반+프랜차이즈)이 깨진 행만 비율(%)->개수로 역산
전체_보정후_예상값 = (df['일반_점포_수'] * df['프랜차이즈_점포_수'] / 100).round(0).astype(int)
원래_공식_성립 = df['전체_점포_수'] == (df['일반_점포_수'] + df['프랜차이즈_점포_수'])
df['프랜차이즈_점포_수_보정'] = np.where(원래_공식_성립, df['프랜차이즈_점포_수'], 전체_보정후_예상값)

# 일반_점포_수_보정, 전체_점포_수_보정도 같은 순서로 재계산
df['일반_점포_수_보정'] = (df['전체_점포_수'] - df['프랜차이즈_점포_수_보정']).clip(lower=0)
전체_후보 = df['일반_점포_수_보정'] + df['프랜차이즈_점포_수_보정']
df['전체_점포_수_보정'] = np.maximum(df['전체_점포_수'], 전체_후보)

print('전체_점포_수_보정 준비 완료')

전체_점포_수_보정 준비 완료


## STEP 2. 개업_점포_수 "이상 행" 정의 및 비중 계산

이상 행 = 개업_점포_수가 전체_점포_수_보정보다 큰 행 (그 분기에 새로 생긴 점포가 현재 전체 점포 수보다 많다는 건 상식적으로 말이 안 됨 → 비율(%)이 잘못 들어간 것으로 의심되는 행)

In [3]:
이상행 = df['개업_점포_수'] > df['전체_점포_수_보정']

print(f"이상 행 개수: {이상행.sum():,} / {len(df):,} ({이상행.sum()/len(df)*100:.2f}%)")
print()

전체_개업_합계 = df['개업_점포_수'].sum()
이상행_개업_합계 = df.loc[이상행, '개업_점포_수'].sum()

print(f"전체 개업_점포_수 합계: {전체_개업_합계:,}")
print(f"이상 행들만의 개업_점포_수 합계: {이상행_개업_합계:,}")
print(f"비중: {이상행_개업_합계 / 전체_개업_합계 * 100:.1f}%")
print()
print("=> 행 개수로는 5.29%뿐인데, 개업_점포_수 '합계'로는 88.7%를 차지함")
print("   -> 소수 행에 극단적으로 큰 값이 몰려있다는 뜻")

이상 행 개수: 16,113 / 304,775 (5.29%)

전체 개업_점포_수 합계: 711,509
이상 행들만의 개업_점포_수 합계: 631,013
비중: 88.7%

=> 행 개수로는 5.29%뿐인데, 개업_점포_수 '합계'로는 88.7%를 차지함
   -> 소수 행에 극단적으로 큰 값이 몰려있다는 뜻


## STEP 3. 이상 행 실제 사례 확인

전체_점포_수가 1~2개인데 개업_점포_수가 300~450개로 찍힌 행들 — 비율(%)이 그대로 개수 칸에 들어간 것으로 의심되는 전형적 사례.

In [4]:
cols = ['상권_코드_명', '서비스_업종_코드_명', '전체_점포_수', '전체_점포_수_보정',
        '일반_점포_수', '프랜차이즈_점포_수', '개업_점포_수', '폐업_점포_수']

df.loc[이상행, cols].sort_values('개업_점포_수', ascending=False).head(10)

,상권_코드_명,서비스_업종_코드_명,전체_점포_수,전체_점포_수_보정,일반_점포_수,프랜차이즈_점포_수,개업_점포_수,폐업_점포_수
29403,염창무학아파트,전자상거래업,2,2,2,0,450,0
124659,중림동,스포츠 강습,1,1,1,0,400,0
72163,송화벽화시장(송화골목시장),전자상거래업,1,1,1,0,400,0
262769,대림1동주민센터,치킨전문점,0,1,1,100,300,1
115059,서울대입구역 8번,일식음식점,1,1,1,0,300,0
264251,노량진역 6번,양식음식점,2,2,2,0,300,0
48662,북창동(시청역_6번),전자상거래업,1,1,1,0,300,0
37314,난우초등학교,편의점,1,1,1,0,300,0
304014,장신구 특화거리 상점가,전자상거래업,1,1,1,0,300,0
250839,경의중앙 신촌역,편의점,0,1,1,100,300,1


## 결론

- 이상 행은 5.29%뿐이지만 개업_점포_수 합계의 88.7%를 차지 → 이 소수 행에 공식 보정(비율→개수 역산)을 적용하면 전체 집계 결과가 이 소수 행에 좌우됨
- 실제로 02번 노트북에서 공식 보정을 시험 적용했을 때 위험 상권 판정이 299개→439개로 급변하는 것을 확인했음(대조군 없이 확신하기엔 변화 폭이 너무 큼)
- 그래서 확실한 안전장치(전체_점포_수를 초과하는 값만 클리핑)만 적용하고, 나머지는 원본 값을 그대로 신뢰하기로 결정 — 기획안_설계.md 5번 문서에 이 결론을 반영해야 함